# `wf_internal_transfer` 단계별 Testbed

이 Notebook은 본인 계좌 간 이체 Workflow를 한 번에 실행하고 결과만 보는 용도가 아니다. 각 단계에서 다음 내용을 차례대로 확인한다.

1. Agent가 받은 사용자 발화
2. Agent 내부 계좌 힌트·금액 Slot 추출 결과
3. Agent가 Backend Tool API에 보낸 요청과 Backend 응답 (출금·입금 계좌 확인 2회)
4. Agent가 Backend로 보낸 계좌 선택 UI Webhook
5. Backend가 검증하여 Agent에 전달한 Resume 값
6. Prepare → 승인 → 인증 → 실행까지 이어지는 나머지 Step

기본 Scenario는 **출금·입금 계좌 둘 다 힌트가 없어 순서대로 사용자 선택이 필요한 경우**다. 잔액조회보다 Step이 많아 중단 지점이 4번(계좌 선택 2회, 승인, 인증) 나온다. Cell을 위에서부터 하나씩 실행한다.

## 전체 실행 흐름

```text
사용자 발화 ("10만원 이체해줘")
  → [Agent 내부] 계좌 힌트·금액 추출 (힌트 없음)
  → [Agent → Backend API] 출금 계좌 확인 → selection_required
  → [Agent → Backend Webhook] 출금 계좌 선택 UI 요청
  → Workflow 중단 (1차)
  → [Backend → Agent] 검증된 from_account_id로 Resume
  → [Agent → Backend API] 입금 계좌 확인 → selection_required
  → [Agent → Backend Webhook] 입금 계좌 선택 UI 요청
  → Workflow 중단 (2차)
  → [Backend → Agent] 검증된 to_account_id로 Resume
  → [Agent → Backend API] 이체 Prepare → ready_for_confirmation
  → [Agent → Backend Webhook] 승인 요청
  → Workflow 중단 (3차)
  → [Backend → Agent] 승인(approved)으로 Resume
  → [Agent → Backend API] 인증 Context 생성 → authentication_required
  → [Agent → Backend Webhook] 인증 요청
  → Workflow 중단 (4차)
  → [Backend → Agent] 인증 성공(verified)으로 Resume
  → [Agent → Backend API] 이체 실행 → completed
  → [Agent → Backend Webhook] 결과 전송
```

Agent는 계좌 소유권·잔액 충분 여부·인증 방법을 직접 판단하지 않고, Backend 응답의 `outcome`을 그대로 Route에 사용한다.

## 0. 공통 설정과 출력 함수

In [1]:
import json
import os

from pydantic import SecretStr

from agent.clients.backend import BackendClientConfig
from agent.testing import MockBackend
from agent.testing.internal_transfer import (
    create_internal_transfer_backend_testbed,
    create_internal_transfer_mock_testbed,
)
from agent.workflow_contracts import WorkflowContractStore
from agent.workflows.internal_transfer import extract_internal_transfer_slots_from_text


def show_json(title, value):
    print(chr(10) + f"--- {title} ---")
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


CHAT_SESSION_ID = "chat_testbed_002"
EXECUTION_CONTEXT_ID = "exec_testbed_002"
THREAD_ID = "thread_internal_testbed_001"

print("공통 설정 완료")

공통 설정 완료


## 1. 관리시트 계약에서 실행 Step 확인

Notebook에서 임의로 Workflow 순서를 정하지 않는다. 관리시트에서 생성한 Manifest를 읽어 현재 Step, 통신 방식과 계약 ID를 확인한다.

In [2]:
store = WorkflowContractStore()
workflow_contract = store.get_workflow("wf_internal_transfer")

step_contracts = [
    {
        "order": step["step_order"],
        "step_id": step["step_id"],
        "mode": step["interaction_mode"],
        "contract_id": step.get("contract_id"),
        "next": step["route_summary"],
    }
    for step in workflow_contract["steps"]
]
show_json("Workflow Step 계약", step_contracts)


--- Workflow Step 계약 ---
[
  {
    "order": 1,
    "step_id": "extract_internal_transfer_slots",
    "mode": "agent_internal",
    "contract_id": null,
    "next": "항상 → resolve_internal_from_account"
  },
  {
    "order": 2,
    "step_id": "resolve_internal_from_account",
    "mode": "backend_tool_api",
    "contract_id": "API-ACCOUNT-LIST",
    "next": "resolved → resolve_internal_to_account | selection_required → request_from_account_selection | no_accounts → emit_internal_from_accounts_empty | 오류 → emit_internal_transfer_error"
  },
  {
    "order": 3,
    "step_id": "request_from_account_selection",
    "mode": "webhook_then_resume",
    "contract_id": "UI-INTERNAL-TRANSFER-FROM-ACCOUNT",
    "next": "selected → resolve_internal_to_account | cancelled → END | 계약 오류 → emit_internal_transfer_error"
  },
  {
    "order": 4,
    "step_id": "emit_internal_from_accounts_empty",
    "mode": "webhook",
    "contract_id": "UI-INTERNAL-TRANSFER-FROM-ACCOUNT",
    "next": "항상 → END"
  },
  

## 2. Step 1 — 사용자 발화에서 Slot 추출

### 입력

사용자가 출금·입금 계좌를 모두 특정하지 않고 금액만 말한다. 이 Step은 Agent 내부에서만 실행되며 API 호출은 없다.

기준 입력은 `10만원 이체해줘`다. 다른 발화의 Slot 결과는 바로 다음 연습 Cell에서 별도로 비교한다.

In [3]:
USER_MESSAGE = "10만원 이체해줘"

slot_input = {"message": USER_MESSAGE}
slot_output = dict(extract_internal_transfer_slots_from_text(USER_MESSAGE))

show_json("Step 1 입력", slot_input)
show_json("Step 1 출력", slot_output)

assert slot_output == {
    "from_account_hint": None,
    "to_account_hint": None,
    "amount": 100000,
}


--- Step 1 입력 ---
{
  "message": "10만원 이체해줘"
}

--- Step 1 출력 ---
{
  "from_account_hint": null,
  "to_account_hint": null,
  "amount": 100000
}


### 다른 발화를 직접 넣어보기

이 Cell은 Workflow 실행에 영향을 주지 않는 연습용이다. 출금·입금 힌트가 하나만 있어도 나머지 값은 살아남는다.

In [4]:
practice_messages = [
    "생활비 통장에서 저축통장으로 10만원 이체해줘",
    "저축 계좌로 10만 원 옮겨줘",
    "이체해줘",
]

for message in practice_messages:
    show_json(message, dict(extract_internal_transfer_slots_from_text(message)))


--- 생활비 통장에서 저축통장으로 10만원 이체해줘 ---
{
  "from_account_hint": "생활비 통장",
  "to_account_hint": "에서 저축통장",
  "amount": 100000
}

--- 저축 계좌로 10만 원 옮겨줘 ---
{
  "from_account_hint": "저축 계좌",
  "to_account_hint": null,
  "amount": 100000
}

--- 이체해줘 ---
{
  "from_account_hint": null,
  "to_account_hint": null,
  "amount": null
}


## 3. Mock Backend 입력값 준비

Backend가 아직 없어도 실제 API와 같은 응답 Schema로 Workflow를 실행한다. 여기서는 출금·입금 계좌 후보를 각각 두 개씩 반환하여 두 번의 `selection_required` Route를 만든다.

아래 값은 Agent가 만든 값이 아니라 **Backend가 소유권과 조회 가능 상태를 검증한 뒤 반환했다고 가정한 값**이다.

In [5]:
def account(account_id, alias, bank_name="신한은행"):
    return {
        "account_id": account_id,
        "bank_name": bank_name,
        "account_alias": alias,
        "account_type": "checking",
        "masked_account_number": "110-***-123456",
        "currency": "KRW",
        "is_default": False,
        "status": "active",
    }


FROM_ACCOUNT_CANDIDATES = [account("acc_001", "생활비 통장"), account("acc_002", "비상금 통장")]
TO_ACCOUNT_CANDIDATES = [account("acc_003", "여행 적금", "토스뱅크"), account("acc_004", "저축 통장", "토스뱅크")]

SELECTED_FROM_ACCOUNT_ID = "acc_001"
SELECTED_TO_ACCOUNT_ID = "acc_003"

show_json("출금 계좌 후보", FROM_ACCOUNT_CANDIDATES)
show_json("입금 계좌 후보", TO_ACCOUNT_CANDIDATES)


--- 출금 계좌 후보 ---
[
  {
    "account_id": "acc_001",
    "bank_name": "신한은행",
    "account_alias": "생활비 통장",
    "account_type": "checking",
    "masked_account_number": "110-***-123456",
    "currency": "KRW",
    "is_default": false,
    "status": "active"
  },
  {
    "account_id": "acc_002",
    "bank_name": "신한은행",
    "account_alias": "비상금 통장",
    "account_type": "checking",
    "masked_account_number": "110-***-123456",
    "currency": "KRW",
    "is_default": false,
    "status": "active"
  }
]

--- 입금 계좌 후보 ---
[
  {
    "account_id": "acc_003",
    "bank_name": "토스뱅크",
    "account_alias": "여행 적금",
    "account_type": "checking",
    "masked_account_number": "110-***-123456",
    "currency": "KRW",
    "is_default": false,
    "status": "active"
  },
  {
    "account_id": "acc_004",
    "bank_name": "토스뱅크",
    "account_alias": "저축 통장",
    "account_type": "checking",
    "masked_account_number": "110-***-123456",
    "currency": "KRW",
    "is_default": false,
    "status":

### Prepare · 인증 · 실행 단계에서 사용할 Mock 응답

승인 화면에 표시할 확인 내역, 인증 요청, 실행 결과를 미리 등록한다.

In [6]:
CONFIRMATION_VIEW = {
    "from_account": {
        "account_id": SELECTED_FROM_ACCOUNT_ID,
        "bank_name": "신한은행",
        "account_alias": "생활비 통장",
        "masked_account_number": "110-***-123456",
    },
    "to_account": {
        "account_id": SELECTED_TO_ACCOUNT_ID,
        "bank_name": "토스뱅크",
        "account_alias": "여행 적금",
        "masked_account_number": "1000-***-654321",
    },
    "amount": 100000,
    "fee": 0,
    "total_debit": 100000,
    "currency": "KRW",
    "expires_at": "2026-07-17T10:05:00+09:00",
}

AUTH_REQUEST_VIEW = {
    "title": "본인 인증이 필요합니다.",
    "available_methods": ["biometric"],
    "expires_at": "2026-07-17T10:10:00+09:00",
}

show_json("승인 화면 확인 내역", CONFIRMATION_VIEW)
show_json("인증 요청 화면", AUTH_REQUEST_VIEW)


--- 승인 화면 확인 내역 ---
{
  "from_account": {
    "account_id": "acc_001",
    "bank_name": "신한은행",
    "account_alias": "생활비 통장",
    "masked_account_number": "110-***-123456"
  },
  "to_account": {
    "account_id": "acc_003",
    "bank_name": "토스뱅크",
    "account_alias": "여행 적금",
    "masked_account_number": "1000-***-654321"
  },
  "amount": 100000,
  "fee": 0,
  "total_debit": 100000,
  "currency": "KRW",
  "expires_at": "2026-07-17T10:05:00+09:00"
}

--- 인증 요청 화면 ---
{
  "title": "본인 인증이 필요합니다.",
  "available_methods": [
    "biometric"
  ],
  "expires_at": "2026-07-17T10:10:00+09:00"
}


## 4. Mock Backend와 실제 Workflow 연결

예상하지 않은 Method나 Path가 호출되면 Mock Backend가 즉시 실패한다. 따라서 등록한 응답과 실제 Workflow 호출이 일치하는지도 함께 검증된다.

In [7]:
mock_backend = MockBackend()
mock_backend.add_success(
    "GET",
    "/api/v1/agent-tools/accounts",
    {
        "account_resolution_outcome": "selection_required",
        "accounts": FROM_ACCOUNT_CANDIDATES,
        "account_ids": [],
    },
)
mock_backend.add_success("POST", "/api/v1/webhooks/agent", {"message_id": "message_from_001"})
mock_backend.add_success(
    "GET",
    "/api/v1/agent-tools/accounts",
    {
        "account_resolution_outcome": "selection_required",
        "accounts": TO_ACCOUNT_CANDIDATES,
        "account_ids": [],
    },
)
mock_backend.add_success("POST", "/api/v1/webhooks/agent", {"message_id": "message_to_001"})
mock_backend.add_success(
    "POST",
    "/api/v1/agent-tools/transfers/internal:prepare",
    {
        "outcome": "ready_for_confirmation",
        "confirmation_id": "confirm_testbed_001",
        "confirmation_view": CONFIRMATION_VIEW,
    },
)
mock_backend.add_success("POST", "/api/v1/webhooks/agent", {"message_id": "message_approval_001"})
mock_backend.add_success(
    "POST",
    "/api/v1/agent-tools/auth-contexts",
    {
        "outcome": "authentication_required",
        "auth_context_id": "auth_testbed_001",
        "auth_request_view": AUTH_REQUEST_VIEW,
    },
)
mock_backend.add_success("POST", "/api/v1/webhooks/agent", {"message_id": "message_auth_001"})
mock_backend.add_success(
    "POST",
    "/api/v1/agent-tools/transfers/internal",
    {
        "outcome": "completed",
        "transaction_id": "txn_testbed_001",
        "completed_at": "2026-07-17T10:11:00+09:00",
    },
)
mock_backend.add_success("POST", "/api/v1/webhooks/agent", {"message_id": "message_result_001"})

mock_config = BackendClientConfig(
    base_url="http://backend.test",
    agent_service_token=SecretStr("agent-service-token"),
    agent_webhook_secret=SecretStr("agent-webhook-secret"),
    retry_backoff_seconds=0,
)

testbed = create_internal_transfer_mock_testbed(mock_backend, mock_config, thread_id=THREAD_ID)
print("Mock Testbed 연결 완료")

Mock Testbed 연결 완료


## 5. Step 2·3 실행 — 출금 계좌 확인 후 1차 중단

`start()`를 호출하면 다음 중단 지점까지 실행한다. 출금 계좌 후보가 여럿이라 바로 선택 UI에서 멈춘다.

In [8]:
start_request = {
    "message": USER_MESSAGE,
    "request_id": "req_internal_start_001",
    "chat_session_id": CHAT_SESSION_ID,
    "execution_context_id": EXECUTION_CONTEXT_ID,
}
show_json("Workflow 시작 입력", start_request)

waiting_from = await testbed.start(**start_request)
show_json("1차 중단 결과", waiting_from.model_dump(mode="json"))

assert waiting_from.status == "waiting"
assert waiting_from.pending_interaction["type"] == "input"
FROM_INPUT_REQUEST_ID = waiting_from.pending_interaction["input_request_id"]


--- Workflow 시작 입력 ---
{
  "message": "10만원 이체해줘",
  "request_id": "req_internal_start_001",
  "chat_session_id": "chat_testbed_002",
  "execution_context_id": "exec_testbed_002"
}

--- 1차 중단 결과 ---
{
  "agent_thread_id": "thread_internal_testbed_001",
  "status": "waiting",
  "pending_interaction": {
    "type": "input",
    "workflow_id": "wf_internal_transfer",
    "step_id": "request_from_account_selection",
    "ui_contract_id": "UI-INTERNAL-TRANSFER-FROM-ACCOUNT",
    "input_request_id": "input_585962e06a6244d2ac2f39e52c91aebc",
    "confirmation_id": null,
    "auth_context_id": null
  },
  "webhook_message_id": "message_from_001",
  "replayed": false
}


### 실제 출금 계좌 확인 API 요청과 선택 UI Webhook

In [9]:
from_account_exchange = mock_backend.exchange_timeline(include_payload=True)[0]
show_json("Agent → Backend 출금 계좌 확인 요청/응답", from_account_exchange)

from_input_webhook = mock_backend.exchange_timeline(include_payload=True)[1]["request"]
show_json("Agent → Backend 출금 계좌 선택 Webhook", from_input_webhook)

assert from_account_exchange["request"].get("account_hint") is None
assert from_input_webhook["metadata"]["step_id"] == "request_from_account_selection"
assert len(from_input_webhook["metadata"]["ui"]["payload"]["accounts"]) == 2


--- Agent → Backend 출금 계좌 확인 요청/응답 ---
{
  "method": "GET",
  "path": "/api/v1/agent-tools/accounts",
  "status_code": 200,
  "request": {
    "account_capability": "withdraw",
    "resolve_selection": "true",
    "all_accounts_requested": "false",
    "limit": "20"
  },
  "response": {
    "success": true,
    "message": "처리 완료",
    "data": {
      "account_resolution_outcome": "selection_required",
      "accounts": [
        {
          "account_id": "acc_001",
          "bank_name": "신한은행",
          "account_alias": "생활비 통장",
          "account_type": "checking",
          "masked_account_number": "110-***-123456",
          "currency": "KRW",
          "is_default": false,
          "status": "active"
        },
        {
          "account_id": "acc_002",
          "bank_name": "신한은행",
          "account_alias": "비상금 통장",
          "account_type": "checking",
          "masked_account_number": "110-***-123456",
          "currency": "KRW",
          "is_default": false,
      

## 6. Step 4·5 실행 — 입금 계좌 확인 후 2차 중단

Backend가 검증한 `from_account_id`로 Resume하면, Agent는 곧바로 입금 계좌 확인 API를 호출한다.

In [10]:
resume_from_request = {
    "agent_thread_id": THREAD_ID,
    "request_id": "req_internal_resume_from_001",
    "chat_session_id": CHAT_SESSION_ID,
    "execution_context_id": EXECUTION_CONTEXT_ID,
    "input_request_id": FROM_INPUT_REQUEST_ID,
    "value": {"account_selection_outcome": "selected", "account_ids": [SELECTED_FROM_ACCOUNT_ID]},
}
show_json("Backend → Agent 출금 계좌 Resume", resume_from_request)

waiting_to = await testbed.resume_input(**resume_from_request)
show_json("2차 중단 결과", waiting_to.model_dump(mode="json"))

assert waiting_to.status == "waiting"
TO_INPUT_REQUEST_ID = waiting_to.pending_interaction["input_request_id"]


--- Backend → Agent 출금 계좌 Resume ---
{
  "agent_thread_id": "thread_internal_testbed_001",
  "request_id": "req_internal_resume_from_001",
  "chat_session_id": "chat_testbed_002",
  "execution_context_id": "exec_testbed_002",
  "input_request_id": "input_585962e06a6244d2ac2f39e52c91aebc",
  "value": {
    "account_selection_outcome": "selected",
    "account_ids": [
      "acc_001"
    ]
  }
}

--- 2차 중단 결과 ---
{
  "agent_thread_id": "thread_internal_testbed_001",
  "status": "waiting",
  "pending_interaction": {
    "type": "input",
    "workflow_id": "wf_internal_transfer",
    "step_id": "request_to_account_selection",
    "ui_contract_id": "UI-INTERNAL-TRANSFER-TO-ACCOUNT",
    "input_request_id": "input_061e90f00c7e4f2abd214a6c8e96dc12",
    "confirmation_id": null,
    "auth_context_id": null
  },
  "webhook_message_id": "message_to_001",
  "replayed": false
}


### 실제 입금 계좌 확인 API 요청과 선택 UI Webhook

In [11]:
to_account_exchange = mock_backend.exchange_timeline(include_payload=True)[2]
show_json("Agent → Backend 입금 계좌 확인 요청/응답", to_account_exchange)

to_input_webhook = mock_backend.exchange_timeline(include_payload=True)[3]["request"]
show_json("Agent → Backend 입금 계좌 선택 Webhook", to_input_webhook)

assert to_input_webhook["metadata"]["step_id"] == "request_to_account_selection"


--- Agent → Backend 입금 계좌 확인 요청/응답 ---
{
  "method": "GET",
  "path": "/api/v1/agent-tools/accounts",
  "status_code": 200,
  "request": {
    "account_capability": "deposit",
    "resolve_selection": "true",
    "all_accounts_requested": "false",
    "exclude_account_ids": "acc_001",
    "limit": "20"
  },
  "response": {
    "success": true,
    "message": "처리 완료",
    "data": {
      "account_resolution_outcome": "selection_required",
      "accounts": [
        {
          "account_id": "acc_003",
          "bank_name": "토스뱅크",
          "account_alias": "여행 적금",
          "account_type": "checking",
          "masked_account_number": "110-***-123456",
          "currency": "KRW",
          "is_default": false,
          "status": "active"
        },
        {
          "account_id": "acc_004",
          "bank_name": "토스뱅크",
          "account_alias": "저축 통장",
          "account_type": "checking",
          "masked_account_number": "110-***-123456",
          "currency": "KRW",
  

## 7. Step 6·7 실행 — Prepare 후 승인 대기(3차 중단)

두 계좌가 모두 확정되면 Agent는 이체 Prepare API를 호출하고, 승인 화면 Webhook을 보낸 뒤 멈춘다.

In [12]:
resume_to_request = {
    "agent_thread_id": THREAD_ID,
    "request_id": "req_internal_resume_to_001",
    "chat_session_id": CHAT_SESSION_ID,
    "execution_context_id": EXECUTION_CONTEXT_ID,
    "input_request_id": TO_INPUT_REQUEST_ID,
    "value": {"account_selection_outcome": "selected", "account_ids": [SELECTED_TO_ACCOUNT_ID]},
}
show_json("Backend → Agent 입금 계좌 Resume", resume_to_request)

waiting_approval = await testbed.resume_input(**resume_to_request)
show_json("3차 중단 결과(승인 대기)", waiting_approval.model_dump(mode="json"))

assert waiting_approval.status == "waiting"
assert waiting_approval.pending_interaction["type"] == "approval"
CONFIRMATION_ID = waiting_approval.pending_interaction["confirmation_id"]


--- Backend → Agent 입금 계좌 Resume ---
{
  "agent_thread_id": "thread_internal_testbed_001",
  "request_id": "req_internal_resume_to_001",
  "chat_session_id": "chat_testbed_002",
  "execution_context_id": "exec_testbed_002",
  "input_request_id": "input_061e90f00c7e4f2abd214a6c8e96dc12",
  "value": {
    "account_selection_outcome": "selected",
    "account_ids": [
      "acc_003"
    ]
  }
}

--- 3차 중단 결과(승인 대기) ---
{
  "agent_thread_id": "thread_internal_testbed_001",
  "status": "waiting",
  "pending_interaction": {
    "type": "approval",
    "workflow_id": "wf_internal_transfer",
    "step_id": "request_internal_transfer_approval",
    "ui_contract_id": "UI-INTERNAL-TRANSFER-CONFIRMATION",
    "input_request_id": null,
    "confirmation_id": "confirm_testbed_001",
    "auth_context_id": null
  },
  "webhook_message_id": "message_approval_001",
  "replayed": false
}


In [13]:
prepare_exchange = next(
    exchange
    for exchange in mock_backend.exchange_timeline(include_payload=True)
    if exchange["path"] == "/api/v1/agent-tools/transfers/internal:prepare"
)
show_json("Agent → Backend Prepare 요청/응답", prepare_exchange)

assert prepare_exchange["request"]["from_account_id"] == SELECTED_FROM_ACCOUNT_ID
assert prepare_exchange["request"]["to_account_id"] == SELECTED_TO_ACCOUNT_ID
assert prepare_exchange["request"]["amount"] == 100000


--- Agent → Backend Prepare 요청/응답 ---
{
  "method": "POST",
  "path": "/api/v1/agent-tools/transfers/internal:prepare",
  "status_code": 200,
  "request": {
    "from_account_id": "acc_001",
    "to_account_id": "acc_003",
    "amount": 100000,
    "currency": "KRW"
  },
  "response": {
    "success": true,
    "message": "처리 완료",
    "data": {
      "outcome": "ready_for_confirmation",
      "confirmation_id": "confirm_testbed_001",
      "confirmation_view": {
        "from_account": {
          "account_id": "acc_001",
          "bank_name": "신한은행",
          "account_alias": "생활비 통장",
          "masked_account_number": "110-***-123456"
        },
        "to_account": {
          "account_id": "acc_003",
          "bank_name": "토스뱅크",
          "account_alias": "여행 적금",
          "masked_account_number": "1000-***-654321"
        },
        "amount": 100000,
        "fee": 0,
        "total_debit": 100000,
        "currency": "KRW",
        "expires_at": "2026-07-17T10:05:00+09:00

## 8. Step 8·9 실행 — 승인 후 인증 대기(4차 중단)

승인(`approved`)으로 Resume하면 Agent는 인증 Context를 생성하고 인증 요청 Webhook을 보낸 뒤 멈춘다.

In [14]:
resume_approval_request = {
    "request_id": "req_internal_resume_approval_001",
    "chat_session_id": CHAT_SESSION_ID,
    "execution_context_id": EXECUTION_CONTEXT_ID,
    "resume": {
        "type": "approval",
        "confirmation_id": CONFIRMATION_ID,
        "approval_outcome": "approved",
    },
}
show_json("Backend → Agent 승인 Resume", resume_approval_request)

from agent.runtime.hitl import ExecutionResumeRequest

waiting_auth = await testbed.resume(
    THREAD_ID, ExecutionResumeRequest.model_validate(resume_approval_request)
)
show_json("4차 중단 결과(인증 대기)", waiting_auth.model_dump(mode="json"))

assert waiting_auth.status == "waiting"
assert waiting_auth.pending_interaction["type"] == "authentication"
AUTH_CONTEXT_ID = waiting_auth.pending_interaction["auth_context_id"]


--- Backend → Agent 승인 Resume ---
{
  "request_id": "req_internal_resume_approval_001",
  "chat_session_id": "chat_testbed_002",
  "execution_context_id": "exec_testbed_002",
  "resume": {
    "type": "approval",
    "confirmation_id": "confirm_testbed_001",
    "approval_outcome": "approved"
  }
}

--- 4차 중단 결과(인증 대기) ---
{
  "agent_thread_id": "thread_internal_testbed_001",
  "status": "waiting",
  "pending_interaction": {
    "type": "authentication",
    "workflow_id": "wf_internal_transfer",
    "step_id": "request_internal_authentication",
    "ui_contract_id": "UI-INTERNAL-TRANSFER-AUTH",
    "input_request_id": null,
    "confirmation_id": null,
    "auth_context_id": "auth_testbed_001"
  },
  "webhook_message_id": "message_auth_001",
  "replayed": false
}


## 9. Step 10·11 실행 — 인증 성공 후 실행과 결과

인증 성공(`verified`)으로 Resume하면 Agent는 이체를 실행하고 결과 Webhook을 전송한 뒤 완료한다.

In [15]:
resume_auth_request = {
    "request_id": "req_internal_resume_auth_001",
    "chat_session_id": CHAT_SESSION_ID,
    "execution_context_id": EXECUTION_CONTEXT_ID,
    "resume": {
        "type": "authentication",
        "auth_context_id": AUTH_CONTEXT_ID,
        "auth_status": "verified",
    },
}
show_json("Backend → Agent 인증 Resume", resume_auth_request)

completed = await testbed.resume(
    THREAD_ID, ExecutionResumeRequest.model_validate(resume_auth_request)
)
show_json("최종 결과", completed.model_dump(mode="json"))

assert completed.status == "completed"


--- Backend → Agent 인증 Resume ---
{
  "request_id": "req_internal_resume_auth_001",
  "chat_session_id": "chat_testbed_002",
  "execution_context_id": "exec_testbed_002",
  "resume": {
    "type": "authentication",
    "auth_context_id": "auth_testbed_001",
    "auth_status": "verified"
  }
}

--- 최종 결과 ---
{
  "agent_thread_id": "thread_internal_testbed_001",
  "status": "completed",
  "pending_interaction": null,
  "webhook_message_id": null,
  "replayed": false
}


In [16]:
execute_exchange = next(
    exchange
    for exchange in mock_backend.exchange_timeline(include_payload=True)
    if exchange["path"] == "/api/v1/agent-tools/transfers/internal"
    and exchange["method"] == "POST"
)
show_json("Agent → Backend 실행 요청/응답", execute_exchange)

result_webhook = mock_backend.exchange_timeline(include_payload=True)[-1]["request"]
show_json("Agent → Backend 결과 Webhook", result_webhook)

assert result_webhook["event_type"] == "component"
assert result_webhook["metadata"]["ui"]["payload"]["transaction_id"] == "txn_testbed_001"


--- Agent → Backend 실행 요청/응답 ---
{
  "method": "POST",
  "path": "/api/v1/agent-tools/transfers/internal",
  "status_code": 200,
  "request": {
    "confirmation_id": "confirm_testbed_001",
    "auth_context_id": "auth_testbed_001"
  },
  "response": {
    "success": true,
    "message": "처리 완료",
    "data": {
      "outcome": "completed",
      "transaction_id": "txn_testbed_001",
      "completed_at": "2026-07-17T10:11:00+09:00"
    }
  }
}

--- Agent → Backend 결과 Webhook ---
{
  "chat_session_id": "chat_testbed_002",
  "event_type": "component",
  "content": "이체가 완료되었습니다.",
  "confirmation_id": null,
  "metadata": {
    "workflow_id": "wf_internal_transfer",
    "step_id": "emit_internal_transfer_result",
    "ui_contract_id": "UI-INTERNAL-TRANSFER-RESULT",
    "ui": {
      "type": "transfer_result",
      "payload": {
        "transaction_id": "txn_testbed_001",
        "completed_at": "2026-07-17T10:11:00+09:00",
        "from_account": {
          "account_id": "acc_001",
     

### 완료 시점의 Agent State

두 계좌, 금액, 거래 ID가 모두 반영되고 중간에 쓰인 `input_request_id`는 제거되어야 한다.

In [17]:
completed_state = await testbed.state(THREAD_ID)
show_json("완료 시점 시스템 State", {
    "current_step_id": completed_state.get("current_step_id"),
    "route_key": completed_state.get("route_key"),
    "status": completed_state.get("status"),
})
show_json("완료 시점 Workflow Data", completed_state["data"])
assert completed_state["data"]["from_account_id"] == SELECTED_FROM_ACCOUNT_ID
assert completed_state["data"]["to_account_id"] == SELECTED_TO_ACCOUNT_ID
assert completed_state["data"]["input_request_id"] is None


--- 완료 시점 시스템 State ---
{
  "current_step_id": "emit_internal_transfer_result",
  "route_key": "completed",
  "status": "completed"
}

--- 완료 시점 Workflow Data ---
{
  "from_account_hint": null,
  "to_account_hint": null,
  "amount": 100000,
  "account_resolution_outcome": "selection_required",
  "accounts": [
    {
      "account_id": "acc_003",
      "bank_name": "토스뱅크",
      "account_alias": "여행 적금",
      "account_type": "checking",
      "masked_account_number": "110-***-123456",
      "currency": "KRW",
      "is_default": false,
      "status": "active"
    },
    {
      "account_id": "acc_004",
      "bank_name": "토스뱅크",
      "account_alias": "저축 통장",
      "account_type": "checking",
      "masked_account_number": "110-***-123456",
      "currency": "KRW",
      "is_default": false,
      "status": "active"
    }
  ],
  "input_request_id": null,
  "account_selection_outcome": "selected",
  "from_account_id": "acc_001",
  "to_account_id": "acc_003",
  "prepare_attempt": 1,
 

## 10. 전체 통신 순서와 자동 확인

마지막으로 Workflow가 실제로 호출한 모든 HTTP 요청을 순서대로 확인한다. 인증 Token과 Secret Header는 출력하지 않는다.

In [18]:
timeline = testbed.request_timeline()
show_json("Agent HTTP 호출 순서", timeline)

paths = [item["path"] for item in timeline]
checks = {
    "출금 계좌 확인 API 1회": paths.count("/api/v1/agent-tools/accounts") == 2,
    "Prepare API 1회": paths.count("/api/v1/agent-tools/transfers/internal:prepare") == 1,
    "Auth Context 생성 1회": paths.count("/api/v1/agent-tools/auth-contexts") == 1,
    "실행 API 1회": paths.count("/api/v1/agent-tools/transfers/internal") == 1,
    "결과 component Webhook 전송": timeline[-1].get("event_type") == "component",
    "Workflow 완료": completed.status == "completed",
}
show_json("자동 확인 결과", checks)
assert all(checks.values())


--- Agent HTTP 호출 순서 ---
[
  {
    "method": "GET",
    "path": "/api/v1/agent-tools/accounts",
    "request_id": "req_internal_start_001:resolve_internal_from_account",
    "execution_context_id": "exec_testbed_002",
    "query_keys": [
      "account_capability",
      "all_accounts_requested",
      "limit",
      "resolve_selection"
    ]
  },
  {
    "method": "POST",
    "path": "/api/v1/webhooks/agent",
    "request_id": "req_internal_start_001",
    "execution_context_id": "exec_testbed_002",
    "event_type": "need_input",
    "step_id": "request_from_account_selection"
  },
  {
    "method": "GET",
    "path": "/api/v1/agent-tools/accounts",
    "request_id": "req_internal_resume_from_001:resolve_internal_to_account",
    "execution_context_id": "exec_testbed_002",
    "query_keys": [
      "account_capability",
      "all_accounts_requested",
      "exclude_account_ids",
      "limit",
      "resolve_selection"
    ]
  },
  {
    "method": "POST",
    "path": "/api/v1/webho

## 11. 다른 Route는 무엇이 달라지는가

| 지점 | Backend 응답/사용자 선택 | 다음 동작 |
| --- | --- | --- |
| 계좌 확인 | `resolved` | 선택 UI 없이 바로 다음 단계 |
| 계좌 확인 | `no_accounts` | 빈 계좌 UI를 전송하고 종료 |
| Prepare | `correction_required` (대상 1개) | 선택 화면 없이 바로 해당 항목 재입력 |
| Prepare | `correction_required` (대상 2개 이상) | 정정 대상 선택 화면 표시 |
| Prepare | `blocked` | 차단 안내 후 종료 |
| 승인 | `cancelled` | 인증·실행 없이 종료 |
| 인증 | `failed`/`expired` | 재시도 선택 화면 표시 |
| 실행 | `reauthentication_required` | Prepare·승인 없이 인증만 재수행 |

이 Route들은 `agent/tests/test_internal_transfer_reference_workflow.py`에서 모두 자동 검증한다(11개 Scenario). Notebook은 Step이 가장 많이 이어지는 "계좌 2회 선택 → 승인 → 인증 → 실행" 경로를 기준 Scenario로 사용한다.

## 12. 실제 Backend로 전환하기

아래 Cell은 기본적으로 실행하지 않는다. 실제 Backend 테스트 Context가 준비되면 환경변수를 설정하고 `RUN_REAL_BACKEND = True`로 변경한다.

이 단계는 Agent Tool API와 Webhook까지 검증한다. Frontend 입력과 Backend 검증을 포함한 전체 E2E는 Backend 실행 시작·Resume Endpoint와 브라우저 테스트에서 확인한다.

In [19]:
RUN_REAL_BACKEND = False

if RUN_REAL_BACKEND:
    required_variables = [
        "BACKEND_BASE_URL",
        "AGENT_SERVICE_TOKEN",
        "AGENT_WEBHOOK_SECRET",
        "TESTBED_CHAT_SESSION_ID",
        "TESTBED_EXECUTION_CONTEXT_ID",
    ]
    missing = [name for name in required_variables if not os.getenv(name)]
    if missing:
        raise RuntimeError(f"Backend Mode 환경변수가 없습니다: {missing}")

    real_config = BackendClientConfig(
        base_url=os.environ["BACKEND_BASE_URL"],
        agent_service_token=SecretStr(os.environ["AGENT_SERVICE_TOKEN"]),
        agent_webhook_secret=SecretStr(os.environ["AGENT_WEBHOOK_SECRET"]),
    )
    real_testbed = create_internal_transfer_backend_testbed(
        real_config, thread_id="thread_internal_real_001"
    )
    real_start = await real_testbed.start(
        message=USER_MESSAGE,
        request_id="req_internal_real_001",
        chat_session_id=os.environ["TESTBED_CHAT_SESSION_ID"],
        execution_context_id=os.environ["TESTBED_EXECUTION_CONTEXT_ID"],
    )
    show_json("실제 Backend 시작 결과", real_start.model_dump(mode="json"))
    await real_testbed.aclose()
else:
    print("Mock Backend Mode — 실제 Backend 호출 없음")

Mock Backend Mode — 실제 Backend 호출 없음


## 13. Testbed 종료

In [20]:
await testbed.aclose()
print("본인이체 단계별 Testbed 종료 완료")

본인이체 단계별 Testbed 종료 완료
